In [1]:
import geopandas as gpd
from shapely import geometry
import pandas as pd
from shapely import wkt
from tqdm.notebook import tqdm  # 导入 tqdm
from shapely.geometry import Point

In [ ]:
from py2neo import Node, Relationship, Graph, NodeMatcher, RelationshipMatcher, Subgraph
import os
import json
g = Graph('bolt://localhost:7687',auth=('neo4j','your password'), name = 'trkg')

In [3]:
from pathlib import Path
from pyproj import CRS

# ---------- locate project root ----------
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "Main data").exists():
            return parent
    raise RuntimeError("Project root (CODE UPLOAD) not found.")

PROJECT_ROOT = find_project_root()
MAIN_DATA = PROJECT_ROOT / "Main data"

# ---------- relative path ----------
crs_path = MAIN_DATA / "roadcrs.txt"

# ---------- read CRS ----------
with open(crs_path, "r", encoding="utf-8") as file:
    crs_text = file.read().strip()

roadcrs = CRS.from_string(crs_text)
roadcrs


<Derived Projected CRS: PROJCS["CGCS2000_3-degree_Gauss-Kruger_CM_120E",GE ...>
Name: CGCS2000_3-degree_Gauss-Kruger_CM_120E
Axis Info [cartesian]:
- [east]: Easting (metre)
- [north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unnamed
- method: Transverse Mercator
Datum: China 2000
- Ellipsoid: CGCS2000
- Prime Meridian: Greenwich

In [8]:
# 查询参数
vehicle_name = 13829
batch = "batch_163"

# 构建 Cypher 查询
cypher_query = """
MATCH (tp:trajectory_point)
WHERE tp.vehicle = $vehicle_name AND tp.batch = $batch
RETURN tp.batch AS batch, tp.name AS name, tp.vehicle AS vehicle, tp.geometry AS geometry, tp.datetime AS datetime
"""

# 执行查询，并将结果转换为 pandas DataFrame
result = g.run(cypher_query, vehicle_name=vehicle_name, batch=batch)

# 将结果转换为 DataFrame
df = pd.DataFrame(result.data())

# 将 'geometry' 列中的 WKT 字符串转换为 Shapely 几何对象
df['geometry'] = df['geometry'].apply(wkt.loads)
# 转换为 GeoDataFrame
df = gpd.GeoDataFrame(df, geometry='geometry', crs=roadcrs)

# 复制 'name' 列作为新的索引
df['name_copy'] = df['name']

# 设置 'name_copy' 列为索引
df.set_index('name_copy', inplace=True)

# 按照索引排序（默认升序）
df_sorted = df.sort_index(ascending=True)  # 可以设置为 False 以降序排序

# 将原始坐标系转换为 WGS84 坐标系（EPSG:4326）
df_sorted_wgs84 = df_sorted.to_crs(epsg=4326)

# 提取纬度和经度作为新列
df_sorted_wgs84['lat'] = df_sorted_wgs84.geometry.y
df_sorted_wgs84['lng'] = df_sorted_wgs84.geometry.x

# 查看转换后的 GeoDataFrame
print(df_sorted_wgs84)

# 使用 tqdm 创建进度条
with tqdm(total=len(df_sorted_wgs84), desc="Creating trajectory_geolocation nodes", unit="row") as pbar:
    for _, row in df_sorted_wgs84.iterrows():
        # 获取每一行的相关信息
        name = row['name']  # 或根据需要使用 'name_copy'
        # if name != 12604:  # 只处理 name 为 0 的行
        #     continue
        lat = row['lat']    # 获取 lat 列的值
        lng = row['lng']    # 获取 lng 列的值
        datetime = row['datetime']  # 假设 datetime 存在于 df_sorted_wgs84 中

        # 构造 Cypher 查询，创建 trajectory_geolocation 节点和 has_geoposition 关系
        cypher = """
        MATCH (tp:trajectory_point {vehicle: $vehicle_name, batch: $batch, name: $name})
        CREATE (g:trajectory_geolocation {batch: $batch, name: $name, vehicle: $vehicle_name, datetime: $datetime, lat: $lat, lng: $lng})
        CREATE (tp)-[:has_geoposition]->(g)
        """
        
        # 执行查询
        g.run(cypher, vehicle_name=vehicle_name, batch=batch, name=name, datetime=datetime, lat=lat, lng=lng)
        
        # 更新进度条
        pbar.update(1)

               batch  name  vehicle                    geometry  \
name_copy                                                         
2915       batch_163  2915    13829  POINT (121.45709 31.22561)   
2916       batch_163  2916    13829  POINT (121.45781 31.22562)   
2917       batch_163  2917    13829  POINT (121.45836 31.22561)   
2918       batch_163  2918    13829  POINT (121.45922 31.22554)   
2919       batch_163  2919    13829  POINT (121.46065 31.22559)   
2920       batch_163  2920    13829  POINT (121.46184 31.22553)   
2921       batch_163  2921    13829  POINT (121.46222 31.22542)   
2922       batch_163  2922    13829  POINT (121.46231 31.22534)   
2923       batch_163  2923    13829  POINT (121.46244 31.22542)   
2924       batch_163  2924    13829  POINT (121.46264 31.22560)   
2925       batch_163  2925    13829  POINT (121.46326 31.22582)   
2926       batch_163  2926    13829  POINT (121.46415 31.22586)   
2927       batch_163  2927    13829  POINT (121.46415 31.22594

Creating trajectory_geolocation nodes:   0%|          | 0/29 [00:00<?, ?row/s]